# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*# Ranked Actions + Reason Codes

The model ranks content pages according to their refresh priority score.

| Rank | Recommended Action | Reason Code |
|------|--------------------|-------------|
| 1 | Refresh pages with declining trends | Negative Trend Percentage |
| 2 | Improve pages with low CTR | Low CTR |
| 3 | Review pages with poor Average Position | High Average Position |
| 4 | Monitor stable pages | Stable Search Performance |

These recommendations are based on observed historical search performance and are intended to support editorial decision-making..*

In [5]:
import os
import subprocess
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rana4682/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
queue = df[['content_id','ctr','avg_position','trend_pct']].copy()

queue["refresh_priority"] = (
    (100 - queue["ctr"]) * 0.30 +
    queue["avg_position"] * 0.40 +
    (-queue["trend_pct"]) * 0.30
)

queue = queue.sort_values("refresh_priority", ascending=False)

queue.head(10)

,content_id,ctr,avg_position,trend_pct,refresh_priority
28214,content_13bbd72aea33,0.0,118.0,-100.0,107.20
24824,content_638236e8066e,0.0,98.0,-100.0,99.20
26536,content_f616ca0ec5ea,0.0,94.0,-100.0,97.60
22347,content_86748254b6bf,0.0,90.0,-100.0,96.00
21834,content_798197311609,0.0,88.9,-96.0,94.36
23618,content_4ed2bf493735,0.0,83.0,-100.0,93.20
9891,content_933296f93aa5,0.0,88.8,-91.7,93.03
21442,content_584e85b1ef21,0.0,81.8,-100.0,92.72
13664,content_a1c4525abad0,0.0,81.3,-100.0,92.52
12977,content_930c7a86e456,0.0,80.2,-100.0,92.08


## 2. Intended use and limits

*# Intended Use and Limits

This action playbook is intended to help editors prioritize which content pages should be reviewed for refresh.

The recommendations are based only on historical search performance signals.

The model should be used as a decision-support tool rather than an automated decision system.

It cannot guarantee improvements in search rankings and should always be combined with human editorial judgment..*

In [7]:
print("Dataset Size :", df.shape)
print("Priority Queue Size :", len(queue))

Dataset Size : (30000, 44)
Priority Queue Size : 30000


# Human Review + No-Go List

Before taking action, editors should verify:

- Content quality
- Search intent
- Content accuracy
- Business relevance

The following actions should never be automated:

- Publishing updated content
- Deleting pages
- Merging pages
- Rewriting articles
- Making SEO decisions without human review

Human review is required before any content changes are published.

In [8]:
# Human review checklist

review_items = [
    "Content Quality",
    "Search Intent",
    "Content Accuracy",
    "Business Relevance"
]

no_go_actions = [
    "Auto Publish",
    "Auto Delete Pages",
    "Auto Merge Pages",
    "Auto Rewrite Articles",
    "Fully Automated SEO Decisions"
]

print("Human Review Checklist")
for item in review_items:
    print("-", item)

print("\nNo-Go Actions")
for action in no_go_actions:
    print("-", action)

Human Review Checklist
- Content Quality
- Search Intent
- Content Accuracy
- Business Relevance

No-Go Actions
- Auto Publish
- Auto Delete Pages
- Auto Merge Pages
- Auto Rewrite Articles
- Fully Automated SEO Decisions


# Monitoring / Retrain Triggers

The model should be monitored and retrained when:

- CTR distribution changes significantly.
- Average Position changes over time.
- Search trends shift.
- New historical search data becomes available.
- Model performance decreases on new data.

Regular retraining helps maintain recommendation quality.

In [9]:
print("Average CTR :", round(df["ctr"].mean(),2))
print("Average Position :", round(df["avg_position"].mean(),2))
print("Average Trend :", round(df["trend_pct"].mean(),2))

Average CTR : 0.51
Average Position : 16.34
Average Trend : -4.79


# Exports for the Paper

The notebook exports the ranked refresh queue for reuse in the deployed research paper.

The exported queue supports the recommendations section of the final capstone paper.

In [10]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/ranked_refresh_queue.csv",
    index=False
)

print("Export completed successfully.")

Export completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.